# Trajectory Optimization: Final Correctness

**Primary principle:**
TOTAL WORK != OBSERVED WALL-CLOCK != DEPENDENCY CRITICAL PATH.

PARALLELIZATION REDUCES WALL-CLOCK, NOT THE AMOUNT OF WORK.
A SUCCESSFUL POLICY BLOCK IS CONTAINMENT SUCCESS.

This notebook demonstrates the true nature of operational optimization on LLM trajectories.

## Part 1: Baseline Sequential Trajectory

We start with a baseline trajectory from the Northstar EU checkout latency scenario. The baseline has unnecessary work, and runs everything sequentially (each step in its own `execution_group`).

In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath("curriculum/intermediate/06-trajectory-optimization"))

from policy import (
    StepType, ResultStatus, ToolEffect, StepClassification,
    OptimizationType, ToolDefinition, TrajectoryStep, Trajectory,
    TrajectoryEvalContext, TrajectoryMetrics, OptimizationCandidate, 
    OptimizationPlan, OptimizationResult, TrajectoryComparison,
    classify_steps, can_parallelize, is_valid_cache_hit, compute_metrics, 
    optimization_regression_gate, should_stop, compare_trajectories,
    find_optimization_candidates, apply_optimization, evaluate_optimization,
    serialize_args
)

tools = {
    "get_service_health": ToolDefinition(name="get_service_health", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="db"),
    "query_logs": ToolDefinition(name="query_logs", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="logs"),
    "get_deployment": ToolDefinition(name="get_deployment", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="k8s"),
    "restart_service": ToolDefinition(name="restart_service", effect=ToolEffect.WRITE, supports_parallel=False, cacheable=False),
    "get_customer": ToolDefinition(name="get_customer", effect=ToolEffect.READ, supports_parallel=True, cacheable=True),
    "get_orders": ToolDefinition(name="get_orders", effect=ToolEffect.READ, supports_parallel=True, cacheable=True)
}

context = TrajectoryEvalContext(
    expected_outcome="Identify database connection pool exhaustion in EU-West.",
    available_evidence_ids=["health_data_eu", "log_data_eu", "deploy_eu", "customer_1", "order_1"],
    required_evidence_ids=["health_data_eu", "log_data_eu"],
    forbidden_tools=["restart_service"],
    tenant_id="northstar",
    policy_version="1.0",
    authorization_scope="scope:read",
    tools=tools,
    dependency_graph={
        "get_customer_1": ["get_orders_1"]  # Orders depend on Customer
    }
)

baseline = Trajectory(
    run_id="run-baseline-01",
    tenant_id="northstar",
    final_answer="Identify database connection pool exhaustion in EU-West.",
    agent_version="1.0", prompt_version="1.0", model_version="1.0", tool_version="1.0", policy_version="1.0", dataset_version="1.0",
    steps=[
        TrajectoryStep(step_id="step_1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01, execution_group=0),
        TrajectoryStep(step_id="step_2", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["health_data_eu"], latency_ms=10, cost_usd=0, execution_group=1),
        TrajectoryStep(step_id="step_3", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02, execution_group=2),
        TrajectoryStep(step_id="step_4", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["log_data_eu"], latency_ms=10, cost_usd=0, execution_group=3),
        TrajectoryStep(step_id="step_5", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01, execution_group=4),
        TrajectoryStep(step_id="step_6", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["health_data_eu"], latency_ms=10, cost_usd=0, execution_group=5),
        TrajectoryStep(step_id="step_7", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02, execution_group=6),
        TrajectoryStep(step_id="step_8", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["log_data_eu"], latency_ms=10, cost_usd=0, execution_group=7),
        TrajectoryStep(step_id="step_9", step_type=StepType.TOOL_CALL, tool_name="get_deployment", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=600, cost_usd=0.01, execution_group=8),
        TrajectoryStep(step_id="step_10", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["deploy_eu"], latency_ms=10, cost_usd=0, execution_group=9),
    ]
)
print("Baseline trajectory loaded.")

## Part 2 & 3: Instrumentation, Metrics, and Step Classification

Before optimizing, we must classify each step. Note that `total_work_ms` represents sum of all computing time, while `observed_wall_clock_latency_ms` measures the real latency due to scheduling groups.

In [ ]:
classified_baseline = classify_steps(baseline, context.tools)

print("Step Classifications:")
for s in classified_baseline.steps:
    if s.step_type == StepType.TOOL_CALL:
        print(f"  {s.tool_name} -> {s.classification.value}")
        
baseline_metrics = compute_metrics(baseline, context)
print(f"\nTotal Work: {baseline_metrics.total_work_ms}ms")
print(f"Wall Clock Latency (Sequential Schedule): {baseline_metrics.observed_wall_clock_latency_ms}ms")
print(f"Theoretical Critical Path (DAG): {baseline_metrics.dependency_critical_path_ms}ms")


## Part 4: Duplicate Removal

We see duplicate reads. We can safely strip them to lower both total work and wall clock latency.

In [ ]:
candidates = find_optimization_candidates(baseline, context)

# Apply removal
dup_candidate = next(c for c in candidates if c.optimization_type == OptimizationType.REMOVE_DUPLICATE_READ)
optimized_no_dups = apply_optimization(baseline, dup_candidate)

result_no_dups = evaluate_optimization(baseline, optimized_no_dups, dup_candidate, context)
print(f"Duplicate removal accepted? {result_no_dups.accepted}. Latency Saved: {result_no_dups.actual_latency_savings_ms}ms")


## Part 5: Dependency-Aware Parallelism

Parallelizing reads groups them into the same `execution_group`. The `total_work_ms` stays the same, but `observed_wall_clock_latency_ms` decreases!

In [ ]:
# Apply parallelization
par_candidate = next(c for c in candidates if c.optimization_type == OptimizationType.PARALLELIZE_READS)
optimized_parallel = apply_optimization(optimized_no_dups, par_candidate)

result_par = evaluate_optimization(optimized_no_dups, optimized_parallel, par_candidate, context)
print(f"Parallelization accepted? {result_par.accepted}.")
print(f"Total Work remains: {compute_metrics(optimized_parallel, context).total_work_ms}ms")
print(f"Wall Clock Latency Saved: {result_par.actual_latency_savings_ms}ms")


## Part 6: Deterministic Caching

Cache validation must explicitly bind to tenant, authorization scope, policy version, source version, precise tool name, and canonical argument serialization.

In [ ]:
now = time.time()

args_canonical = serialize_args({"service": "checkout"})

# Valid hit
cache_entry = {
    "policy_version": "1.0", 
    "authorization_scope": "scope:read",
    "tool_name": "get_service_health",
    "expires_at": now + 1000, 
    "source_version": "v1.2", 
    "arguments": args_canonical
}
s_test = TrajectoryStep(step_id="c1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=10, cost_usd=0, source_version="v1.2")

print("Valid cache hit?", is_valid_cache_hit(s_test, cache_entry, context, now))
print("Valid hit (tool mismatch)?", is_valid_cache_hit(TrajectoryStep(**{**s_test.model_dump(), "tool_name": "query_logs"}), cache_entry, context, now))
print("Valid hit (auth mismatch)?", is_valid_cache_hit(s_test, {**cache_entry, "authorization_scope": "scope:admin"}, context, now))


## Part 7: Unknown Tool Classification

An unknown tool safely falls back to `UNKNOWN_TOOL_METADATA` so that the optimizer does not accidentally assume it is safe to parallelize, cache, or remove.

In [ ]:
# Unknown tool
s_unknown = TrajectoryStep(step_id="u1", step_type=StepType.TOOL_CALL, tool_name="unknown_beta_feature", target_tenant_id="northstar", latency_ms=10, cost_usd=0)
traj_u = Trajectory(**{**baseline.model_dump(), "steps": [s_unknown]})
classified_u = classify_steps(traj_u, context.tools)
print(f"Unknown Tool Classification: {classified_u.steps[0].classification.value}")
print(f"Can parallelize? {can_parallelize(s_test, s_unknown, context)}")


## Part 8: Policy Block Semantics

A forbidden tool attempt that is blocked is NOT a policy violation. It is a containment success!

In [ ]:
s_forbidden = TrajectoryStep(step_id="f1", step_type=StepType.TOOL_CALL, tool_name="restart_service", target_tenant_id="northstar", latency_ms=10, cost_usd=0)
s_block = TrajectoryStep(step_id="f2", step_type=StepType.POLICY_DECISION, result_status=ResultStatus.POLICY_BLOCKED, latency_ms=10, cost_usd=0)

traj_block = Trajectory(**{**baseline.model_dump(), "steps": [s_forbidden, s_block]})
m_block = compute_metrics(traj_block, context)
print(f"Attempted forbidden? {m_block.forbidden_attempted}")
print(f"Executed forbidden? {m_block.forbidden_executed}")
print(f"Policy Compliant? {m_block.is_policy_compliant}")
